In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.mathtext import _mathtext as mathtext
from scipy.stats import linregress
from sklearn.metrics import mean_absolute_error, mean_squared_error

mathtext.FontConstantsBase.sup1 = 0.45
plt.rcParams['font.family'] = 'Helvetica'
plt.rcParams['font.size'] = 16
plt.rcParams['mathtext.fontset'] = 'custom'
plt.rcParams['mathtext.rm'] = 'Helvetica'
plt.rcParams['mathtext.it'] = 'Helvetica:italic'
plt.rcParams['mathtext.bf'] = 'Helvetica:bold'

# Processed output from the original Rovai-validation workflow.
results_df = pd.read_csv('../../data/supplementary_figures/figure_level/figS05_rovai_validation.csv')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plt.subplots_adjust(wspace=0.25, bottom=0.15)

datasets_to_plot = [
    {'y_mod_col': 'OurModel_AGB', 'title': 'a', 'ylabel': 'this study'},
    {'y_mod_col': 'Hu_AGB', 'title': 'b', 'ylabel': 'Hu et al.'},
    {'y_mod_col': 'Simard_AGB', 'title': 'c', 'ylabel': 'Simard et al.'},
]

y_obs = results_df['AGB']

for ax, config in zip(axes, datasets_to_plot):
    y_mod = results_df[config['y_mod_col']]
    valid_indices = y_obs.notna() & y_mod.notna()
    y_obs_clean = y_obs[valid_indices]
    y_mod_clean = y_mod[valid_indices]

    slope, intercept, r_value, _, _ = linregress(y_obs_clean, y_mod_clean)
    r2 = r_value**2
    rmse = np.sqrt(mean_squared_error(y_obs_clean, y_mod_clean))
    mae = mean_absolute_error(y_obs_clean, y_mod_clean)
    pbias = 100 * np.sum(y_mod_clean - y_obs_clean) / np.sum(y_obs_clean)

    ax.scatter(y_obs_clean, y_mod_clean, facecolors='none', edgecolors='gray', s=30)
    max_val = max(y_obs_clean.max(), y_mod_clean.max()) if len(y_obs_clean) > 0 else 100
    lims = [-0.05 * max_val, 1.05 * max_val]
    ax.set(xlim=lims, ylim=lims)
    ax.plot(lims, lims, linestyle='--', color='gray')
    ax.plot(np.array(lims), slope * np.array(lims) + intercept, color='red')
    ax.set_ylabel(f"AGB from {config['ylabel']} (Mg DM ha$^{{-1}}$)")
    ax.set_title(config['title'], loc='left', fontweight='bold', fontsize=16)
    ax.set_aspect('equal', adjustable='box')
    metrics_text = f'R$^2 = {r2:.2f}$\nRMSE = {rmse:.2f}\nMAE = {mae:.2f}\nPBIAS = {pbias:.2f}%'
    ax.text(0.05, 0.95, metrics_text, transform=ax.transAxes, verticalalignment='top')

fig.supxlabel('Observed AGB from Rovai et al. (Mg DM ha$^{-1}$)', fontsize=16)
plt.savefig('../../figures/supplementary/figS05_validation_rovai.png', dpi=400, bbox_inches='tight')
plt.show()
